In [103]:
import cv2
import numpy as np
import time
from sympy import symbols, cos, sin, Matrix
from pymycobot.mycobot import MyCobot

START_POS = [0,0,0,0,0,0]
NEUTRAL = [0, -40, 55, 0, 30, 0]
WATCHING_POS = [88.3, 0, 150, -180, 46.5, -180]
bot = MyCobot('/dev/ttyACM0', 115200)

try:
    mtx = np.array(np.loadtxt('calibration/calib_mtx.csv', delimiter=','))
    dist = np.array(np.loadtxt('calibration/calib_dist.csv', delimiter=','))
except FileNotFoundError:
    print('Files not found, make sure you ran calibration!')

aruco_dict = cv2.aruco.getPredefinedDictionary(
    cv2.aruco.DICT_4X4_50
)

# detector = cv2.aruco.ArucoDetector(aruco_dict)

tagSize = 38 / 1000 # m

def drawSquare(frame, corners):
    for i in range(4):
        cv2.line(frame, (int(corners[0][i][0]),int(corners[0][i][1])), (int(corners[0][(i+1)%4][0]), int(corners[0][(i+1)%4][1])), (0,0,255), 3)

def trans(a, alpha, d, theta):
    return Matrix([
        [cos(theta), -sin(theta), 0, a],
        [sin(theta)*cos(alpha), cos(theta)*cos(alpha), -sin(alpha), -sin(alpha)*d],
        [sin(theta)*sin(alpha), cos(theta)*sin(alpha), cos(alpha), cos(alpha)*d],
        [0, 0, 0, 1]
    ])

qs = symbols('q1:7')
q1, q2, q3, q4, q5, q6 = qs
l0, l1, l2, l3, l4, l5 = [114, 0, 100, 97, 0, 56] # mm
alpha2, alpha4, alpha5, alpha6 = np.radians([-90, -90, 90, -90]) # rad

l3_off = 10 #mm
j2_off = np.radians(-90) # rad

dh = [
    # [0, 0, l0, q1],
    # [0, alpha2, l1, q2 + j2_off],
    # [l2, 0, 0, q3],
    # [l3_off, alpha4, l3, q4],
    # [0, alpha5, l4, q5],
    # [0, alpha6, l5, q6]

    [0, 0, 112, q1],
    [0, -90, 0, q2],
    [108, 0, 0, q3],
    [10, -90, 95, q4],
    [0, 90, 0, q5],
    [0, -90, 57, q6]
]

T_ew = Matrix.eye(4)
for row in dh:
    T_ew *= trans(*row)

T_ec = np.array([
    [0, 0, 1, -15],
    [-1, 0, 0, -45],
    [0, -1, 0, 0],
    [0, 0, 0, 1]
])

Note: This class is no longer maintained since v3.6.0, please refer to the project documentation: https://github.com/elephantrobotics/pymycobot/blob/main/README.md


In [104]:
R_ew = T_ew[0:3, 0:3]
t_ew = T_ew[0:3, 3]
T_we = Matrix.eye(4)
T_we[0:3, 0:3] = R_ew.T
T_we[0:3, 3] = -R_ew.T @ t_ew

In [105]:
bot.send_angles(START_POS, 40)
time.sleep(2)
bot.set_gripper_value(95, 40, 1)
time.sleep(2)
bot.send_angles(NEUTRAL, 40)
time.sleep(2)

In [106]:
cv2.destroyAllWindows()
cv2.waitKey(1)

cap = cv2.VideoCapture(1)
while True:
    ret, frame = cap.read()
    if (not ret): continue

    # corners, ids, _ = detector.detectMarkers(frame)
    corners, ids, _ = cv2.aruco.detectMarkers(frame, aruco_dict)

    objpoints = np.array([
        (-tagSize/2, tagSize/2, 0),
        (tagSize/2, tagSize/2, 0),
        (tagSize/2, -tagSize/2, 0),
        (-tagSize/2, -tagSize/2, 0)
    ])

    key = cv2.waitKey(1)

    if corners is not None and ids is not None:
        for square, id in zip(corners, ids):
            drawSquare(frame, square)
            ret, rvec, tvec = cv2.solvePnP(objpoints, square[0], mtx, dist)
            distance = np.linalg.norm(tvec) 
            t_rounded = np.round(tvec * 1000, 2).flatten().tolist()
            text = ", ".join(map(str, t_rounded))

            cv2.putText(frame, '%f mm' % (distance * 1000), (int(square[0][0][0]), int(square[0][0][1])), cv2.FONT_HERSHEY_COMPLEX, 1, (60,255,70), 2)
            cv2.putText(frame, text, (int(square[0][0][0]), int(square[0][0][1]) + 20), cv2.FONT_HERSHEY_COMPLEX, 0.5, (255,255,255), 2)

            cv2.drawFrameAxes(frame, mtx, dist, rvec, tvec, tagSize * 1.5, 2)
    

    if key == ord(' '):
        bot_angles = bot.get_angles()
        joint_angles = [np.radians(j) for j in (bot_angles if bot_angles != -1 else NEUTRAL)]
        # tvec_4 = np.concatenate((tvec.flatten() * 1000, [1]))
        T_sub = T_ew.subs(zip([q1, q2, q3, q4, q5, q6], joint_angles)).evalf()

        wp_ee = bot.get_coords()
        if wp_ee == -1:
            wp_ee = WATCHING_POS
        
        cp_obj = tvec.flatten() * 1000
        ep_obj = T_ec @ np.concatenate((cp_obj, [1]))
        wp_obj = T_sub @ ep_obj

        # y_off = np.radians(wp_ee[4])
        # T = np.array([
        #     [np.cos(y_off), 0, np.sin(y_off), 0],
        #     [0, 1, 0, 0],
        #     [-np.sin(y_off), 0, np.cos(y_off), 0],
        #     [0, 0, 0, 1]
        # ])
        # obj = T @ obj
        
        # rot = (T @ np.concatenate((rvec.flatten(), [1])))[:-1]
        R_co = cv2.Rodrigues(rvec)[0]
        # z offset in ee frame, y offset in camera frame
        # z_off = -np.arctan2(-rot[2][0], np.sqrt(rot[0][0]**2 + rot[1][0]**2))

        des_pos = wp_ee + np.concatenate((wp_obj[:3], [0,90-wp_ee[4],0]))
        # obj = T_sub @ (T_ec @ tvec_4)
        # ee_pos = np.concatenate((obj[:3], [-160, 90, -160]))
        print('EE: ', wp_ee)
        print('rvec: ', cv2.Rodrigues(rvec)[0])
        # print('z_off: ', np.degrees(z_off))
        print('tvec: ', np.concatenate((tvec.flatten() * 1000, [1])).tolist())
        print('obj in ee: ', ep_obj.tolist())
        print('obj: ', wp_obj.tolist())
        print('des: ', des_pos)
        bot.send_coords(np.concatenate((wp_obj[:-1], [-160, 90, -160])), 40)
        # des_pos[1] += 45
        # des_pos[2] += 125
        # bot.send_coords(des_pos, 40)
        # time.sleep(2)
        # des_pos[2] -= 75
        # bot.send_coords(des_pos, 40)
        # time.sleep(2)
        # bot.set_gripper_value(70, 40)
        # time.sleep(2)
        # bot.send_angles(START_POS, 40)

    cv2.imshow('ArUco', frame)

    if key == ord('r'):
        bot.set_gripper_value(95, 40, 1)
        time.sleep(2)
        bot.send_angles(NEUTRAL, 40)
        time.sleep(2)

    if key == ord('q'):
        break

cap.release()
cv2.destroyAllWindows()
cv2.waitKey(1)

EE:  [88.5, -0.1, 149.4, -179.01, 46.13, -178.93]
rvec:  [[ 0.9996534   0.02601652 -0.00402741]
 [ 0.02001041 -0.65147277  0.75840808]
 [ 0.01710739 -0.7582258  -0.65176757]]
tvec:  [-40.28182527509762, -6.195934069002981, 199.083407365685, 1.0]
obj in ee:  [184.083407365685, -4.718174724902383, 6.195934069002981, 1.0]
obj:  [162.306111826235, -140.966453709021, -24.6701701808352, 1.00000000000000]
des:  [250.806111826235 -141.066453709021 124.729829819165 -179.01 90.0 -178.93]


-1

In [76]:
T_sub[0:3, 3] = [0,0,0]

In [92]:
T_sub

Matrix([
[  -0.690592478409344, -0.00328747860256138,    0.723236628807528,   72.0741139472335],
[-0.00955061186519371,    0.999943929318307, -0.00457427944761236, -0.162480804639242],
[  -0.723181038590919,  -0.0100663153090954,   -0.690585153850456,   136.836222640951],
[                   0,                    0,                    0,                1.0]])

In [88]:
T_sub ** -1

Matrix([
[   0.691674064123896, 0.00393154867452358,  0.722199094393889,   0],
[-0.00598333549845576,  -0.999919669059459, 0.0111738589724029,   0],
[    0.72218501003181, -0.0120498279258288, -0.691594977521028,   0],
[                   0,                   0,                  0, 1.0]])

In [82]:
from scipy.spatial.transform import Rotation as R

In [93]:
r = R.from_matrix(T_sub[0:3,0:3])
r.as_euler('xyz', degrees=True)

array([-179.16488715,   46.31774127, -179.20767329])